In [1]:
!pip install -q langchain==0.1.20 langchain-groq langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
google-adk 2.4.0 requires tenacit

In [2]:
import os
from langchain_groq import ChatGroq

# Paste your Groq API key here between the quotes
os.environ["GROQ_API_KEY"] = "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"

# Connect to Groq AI
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Connected to Groq successfully!")

Connected to Groq successfully!


In [3]:
# Test the connection with a real question
response = llm.invoke("What is fleet orchestration in one sentence?")
print(response.content)

Fleet orchestration refers to the process of managing and coordinating a large number of vehicles, such as trucks, taxis, or delivery vans, to optimize their routes, schedules, and overall performance in real-time, often using data analytics, artificial intelligence, and IoT technologies.


In [4]:
from langchain.tools import tool
from datetime import datetime

# Define fleet data inside the tool so it is always available
fleet_vehicles = [
    {"id": "ZX001", "state": "Available", "battery": 85, "minutes_in_state": 5},
    {"id": "ZX002", "state": "Charging", "battery": 23, "minutes_in_state": 45},
    {"id": "ZX003", "state": "Passenger", "battery": 67, "minutes_in_state": 12},
    {"id": "ZX004", "state": "Offline", "battery": 8, "minutes_in_state": 120},
    {"id": "ZX005", "state": "Available", "battery": 91, "minutes_in_state": 3},
    {"id": "ZX006", "state": "Maintenance", "battery": 45, "minutes_in_state": 200},
    {"id": "ZX007", "state": "Passenger", "battery": 15, "minutes_in_state": 35},
    {"id": "ZX008", "state": "Charging", "battery": 78, "minutes_in_state": 8},
    {"id": "ZX009", "state": "Available", "battery": 55, "minutes_in_state": 2},
    {"id": "ZX010", "state": "Offline", "battery": 3, "minutes_in_state": 300},
]

@tool
def detect_anomalies(fleet_data: str) -> str:
    """Analyzes fleet vehicle states and detects anomalies that need attention."""

    anomalies = []

    for vehicle in fleet_vehicles:
        vid = vehicle["id"]
        state = vehicle["state"]
        battery = vehicle["battery"]
        mins = vehicle["minutes_in_state"]

        if battery < 10:
            anomalies.append(f"{vid}: CRITICAL - Battery at {battery}%, immediate charging required")
        elif battery < 20 and state == "Passenger":
            anomalies.append(f"{vid}: WARNING - Battery at {battery}% with passenger onboard")
        if state == "Offline" and mins > 60:
            anomalies.append(f"{vid}: ALERT - Offline for {mins} minutes, investigate immediately")
        if state == "Maintenance" and mins > 180:
            anomalies.append(f"{vid}: ALERT - In maintenance for {mins} minutes, status update needed")
        if state == "Charging" and battery > 70:
            anomalies.append(f"{vid}: INFO - Charging at {battery}%, charger can be released")

    if not anomalies:
        return "No anomalies detected. Fleet operating normally."

    return "\n".join(anomalies)

@tool
def generate_incident_report(anomalies: str) -> str:
    """Generates a structured incident report with recommended actions for fleet operators."""

    if "No anomalies" in anomalies:
        return "FLEET STATUS REPORT\nTimestamp: " + datetime.now().strftime("%Y-%m-%d %H:%M") + "\nStatus: ALL SYSTEMS NORMAL\nNo operator action required."

    lines = anomalies.strip().split("\n")
    report = f"""
ZOOX FLEET INCIDENT REPORT
===========================
Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M")}
Total Vehicles Monitored: {len(fleet_vehicles)}
Anomalies Found: {len(lines)}

ANOMALIES DETECTED:
{anomalies}

RECOMMENDED ACTIONS:
- CRITICAL vehicles: Dispatch recovery team immediately
- WARNING vehicles: Monitor and prepare for mid-ride charging stop
- ALERT vehicles: Contact operations center for status update
- INFO vehicles: Free up charger for higher priority vehicles

Fleet Health Score: {round((len(fleet_vehicles) - len(lines)) / len(fleet_vehicles) * 100)}%
===========================
"""
    return report

print("Both tools ready!")


Both tools ready!


In [5]:
@tool
def generate_incident_report(anomalies: str) -> str:
    """Generates a structured incident report with recommended actions for fleet operators."""

    if "No anomalies" in anomalies:
        return "FLEET STATUS REPORT\nTimestamp: " + datetime.now().strftime("%Y-%m-%d %H:%M") + "\nStatus: ALL SYSTEMS NORMAL\nNo operator action required."

    report = f"""
ZOOX FLEET INCIDENT REPORT
===========================
Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M")}
Total Vehicles Monitored: {len(vehicles)}

ANOMALIES DETECTED:
{anomalies}

RECOMMENDED ACTIONS:
- CRITICAL vehicles: Dispatch recovery team immediately
- WARNING vehicles: Monitor and prepare for mid-ride charging stop
- ALERT vehicles: Contact operations center for status update
- INFO vehicles: Free up charger for higher priority vehicles

Fleet Health Score: {round((len(vehicles) - len(anomalies.split(chr(10)))) / len(vehicles) * 100)}%
===========================
"""
    return report

print("Incident report tool ready!")

Incident report tool ready!


In [6]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate

# Give the agent its tools
tools = [detect_anomalies, generate_incident_report]

# Define how the agent thinks step by step
template = '''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}'''

prompt = PromptTemplate.from_template(template)

# Create the agent
agent = create_react_agent(llm, tools, prompt)

# Create the executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

print("Zoox Fleet Agent is ready!")


Zoox Fleet Agent is ready!


In [7]:
# Run the Zoox Fleet Agent
result = agent_executor.invoke({
    "input": "Which vehicles should be prioritized for immediate action and why? What would you recommend the operations manager do in the next 10 minutes?"
})

print("\n" + "="*50)
print("FINAL REPORT:")
print("="*50)
print(result["output"])



> Entering new AgentExecutor chain...
Thought: To determine which vehicles should be prioritized for immediate action, I need to analyze the current state of the fleet vehicles and identify any anomalies that require attention. This will help me understand which vehicles are experiencing issues and need to be addressed urgently.

Action: detect_anomalies
Action Input: fleet_data (assuming this is a string containing the current state of the fleet vehicles)ZX004: CRITICAL - Battery at 8%, immediate charging required
ZX004: ALERT - Offline for 120 minutes, investigate immediately
ZX006: ALERT - In maintenance for 200 minutes, status update needed
ZX007: WARNING - Battery at 15% with passenger onboard
ZX008: INFO - Charging at 78%, charger can be released
ZX010: CRITICAL - Battery at 3%, immediate charging required
ZX010: ALERT - Offline for 300 minutes, investigate immediatelyI have detected several anomalies in the fleet vehicles, including critical, alert, warning, and info-level iss

NameError: name 'vehicles' is not defined